In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 4 — Local Resume Rewriter
## Improve Resume Bullets Using STAR Format + Local LLM

**What you'll learn:**
- Parse resume sections (experience, skills, education)
- Rewrite bullets using STAR methodology
- Tailor content to a specific job description
- Score resume-JD fit quantitatively

**Stack:** Ollama · LangChain · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.3)
print("LLM ready!")

## Step 2 — Sample Resume and Job Description

In [ ]:
sample_resume = """
EXPERIENCE
Software Engineer, Acme Corp (2022-2024)
- Worked on backend services
- Helped improve system performance
- Did code reviews and mentored junior developers
- Built APIs for the mobile app

Data Analyst Intern, BigData Inc (2021-2022)
- Analyzed data and made reports
- Used Python and SQL
- Helped with dashboard creation

SKILLS
Python, SQL, Docker, AWS, React, Git

EDUCATION
B.S. Computer Science, State University, 2021
"""

target_jd = """
Senior Backend Engineer — TechCo
Requirements:
- 3+ years building scalable backend services
- Experience with Python, FastAPI, PostgreSQL
- Proven track record of improving system performance (latency, throughput)
- Experience mentoring engineers and leading code reviews
- Familiarity with CI/CD pipelines and containerization (Docker, K8s)
- Strong communication and documentation skills
"""
print("Resume and JD loaded.")

## Step 3 — Rewrite Bullets with STAR Method

In [ ]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert resume coach. Rewrite each resume bullet
using the STAR method (Situation, Task, Action, Result). Make bullets:
- Start with a strong action verb
- Include quantifiable metrics where possible
- Be specific about technologies and impact
- Keep each bullet to 1-2 lines

If a bullet is vague, infer reasonable details to make it concrete."""),
    ("human", """Rewrite these resume bullets in STAR format:

{bullets}

Return each original bullet followed by its improved version.""")
])

rewrite_chain = rewrite_prompt | llm | StrOutputParser()

result = rewrite_chain.invoke({"bullets": sample_resume})
print(result)

## Step 4 — Tailor Resume to Job Description

In [ ]:
tailor_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a resume tailoring expert. Given a resume and target JD:
1. Identify which resume experiences match JD requirements
2. Rewrite bullets to emphasize relevant skills and outcomes
3. Suggest which skills to highlight/add
4. Recommend section ordering for maximum impact"""),
    ("human", """Resume:
{resume}

Target Job Description:
{jd}

Produce a tailored resume with rewritten bullets optimized for this JD.""")
])

tailor_chain = tailor_prompt | llm | StrOutputParser()
tailored = tailor_chain.invoke({"resume": sample_resume, "jd": target_jd})
print(tailored)

## Step 5 — Score Resume-JD Fit

In [ ]:
from pydantic import BaseModel, Field

class FitScore(BaseModel):
    overall_score: int = Field(description="1-100 fit score")
    matched_requirements: list[str] = Field(description="JD requirements matched")
    gaps: list[str] = Field(description="JD requirements not addressed")
    recommendations: list[str] = Field(description="Specific improvements to make")

scoring_llm = llm.with_structured_output(FitScore)
fit = scoring_llm.invoke(
    f"""Score this resume against the job description.

Resume: {sample_resume}

Job Description: {target_jd}

Provide a detailed fit analysis."""
)

print(f"Overall Fit Score: {fit.overall_score}/100")
print(f"\nMatched Requirements ({len(fit.matched_requirements)}):")
for m in fit.matched_requirements:
    print(f"  ✓ {m}")
print(f"\nGaps ({len(fit.gaps)}):")
for g in fit.gaps:
    print(f"  ✗ {g}")
print(f"\nRecommendations:")
for r in fit.recommendations:
    print(f"  → {r}")

## What You Learned
- **STAR method** for resume bullet rewriting
- **JD-resume tailoring** with targeted prompts
- **Structured scoring** with Pydantic outputs
- **Quantitative fit analysis** to prioritize improvements